# Modelo 1: Expected Goals (xG) - Power EDA Edition
**Machine Learning I - Universidad Externado de Colombia**

Este notebook integra el análisis espacial avanzado y el entrenamiento del clasificador logístico para predecir la probabilidad de gol.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from mplsoccer import Pitch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Estilo Premier League
PL_PURPLE = '#38003c'
PL_CYAN = '#00ff85'
PL_PINK = '#e90052'
sns.set_theme(style="darkgrid", palette=[PL_CYAN, PL_PINK])
plt.rcParams['figure.facecolor'] = 'white'

## 1. Power EDA: Visualización Espacial (Shot Map)
Utilizamos la librería `mplsoccer` para proyectar los tiros sobre un campo real y entender la geografía del gol.

In [ ]:
events_df = pd.read_csv('../../data/events.csv')
shots = events_df[events_df['is_shot'] == 1].copy()

pitch = Pitch(pitch_type='opta', pitch_color=PL_PURPLE, line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# Graficar Fallos y Goles
pitch.scatter(shots[shots['is_goal']==0].x, shots[shots['is_goal']==0].y, 
              alpha=0.3, s=20, color='gray', label='Fallos', ax=ax)
pitch.scatter(shots[shots['is_goal']==1].x, shots[shots['is_goal']==1].y, 
              alpha=0.8, s=50, color=PL_CYAN, edgecolors='white', label='GOLES', ax=ax)

ax.set_title('Mapa de Calor de Tiros - Premier League', color=PL_PURPLE, fontsize=18, fontweight='bold')
plt.legend(facecolor='white')
plt.savefig('../../img/shot_map_pl.png', bbox_inches='tight')
plt.show()

## 2. Análisis Bidimensional: Geometría del Peligro
Observamos la densidad de probabilidad cruzando la Distancia al arco con el Ángulo de tiro.

In [ ]:
# Usamos la matriz procesada que ya tiene estas columnas
df_train = pd.read_csv('../../data/xg_train.csv')

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_train, x='dist_al_arco', y='angulo_tiro', hue='is_goal', fill=True, alpha=0.5)
plt.title('Densidad: Distancia vs Ángulo (Goles en Cian)', color=PL_PURPLE, fontweight='bold')
plt.show()

## 3. Benchmarking: Modelo vs. Naive Baseline
En la Premier, promediar "nunca es gol" acierta el 88.8% de las veces. Nuestra meta es superar ese sesgo con inteligencia.

In [ ]:
X = df_train[['dist_al_arco', 'angulo_tiro', 'is_BigChance', 'is_Penalty', 'is_OneOnOne', 'threat']]
y = df_train['is_goal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

acc_model = model.score(X_test, y_test)
acc_naive = 1 - y_test.mean()

print(f"--- COMPARATIVA DE DESEMPEÑO ---")
print(f"Precisión Naive (Frecuencia): {acc_naive:.4f}")
print(f"Precisión de nuestra IA:     {acc_model:.4f}")
print(f"Valor agregado de xG:        {(acc_model - acc_naive)*100:+.2f}%")

## 4. Evaluación Final
Generamos la Curva ROC y Matriz de Confusión con los colores corporativos.

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color=PL_PINK, lw=3, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color=PL_PURPLE)
plt.title('Calidad de Clasificación xG', color=PL_PURPLE, fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('../../img/roc_curve_xg.png')
plt.show()